# Introduction to Generative Models
__ESE 670 Homework 2 (20pt)__

In this notebook, we will explore several basic approaches to image generation using convolutional neural networks. We will generate face photographs. As a reference, we will use aligned, centered celebrity photos from the CelebFaces Attributes (CelebA) dataset.

We will install the necessary packages and download the dataset. The notebook is self-contained: the encoder/decoder architectures are defined below, and the models are trained here, unless your instructor has provided checkpoints in the `checkpoints/` folder, in which case they are loaded.

**Compute.** A GPU is required. With the default settings each model trains in roughly 10–40 minutes on one A100 (the GAN is the slowest). Reduce `MAX_TRAIN_IMAGES` or the epoch counts in the configuration cell for a quicker run.

In [ ]:
!pip install lightning

In [ ]:
import glob
import os
import warnings

import lightning as L
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
import torchvision as tv
import torchvision.transforms.v2 as T
from IPython.display import display
from PIL import Image

In [ ]:
DATA_ROOT = "."                 # where CelebA lives (or will be downloaded)
CKPT_DIR = "checkpoints"        # put AutoEncoder.ckpt / VariationalAutoEncoder.ckpt / GenerativeAdversarialNetwork.ckpt here to skip training
MAX_TRAIN_IMAGES = None         # e.g. 50_000 for faster training; None = the full training split (162,770 images)
MAX_VALID_IMAGES = 5_000        # validation images used during training
GAN_EPOCHS = 50
os.makedirs(CKPT_DIR, exist_ok=True)
ckpt_path = lambda name: os.path.join(CKPT_DIR, f"{name}.ckpt")

In [ ]:
!kaggle datasets download -d jessicali9530/celeba-dataset

In [ ]:
!unzip celeba-dataset.zip

In [ ]:
# CelebA (aligned & cropped faces). We first try the official torchvision download (hosted on Google Drive,
# which sometimes fails because of download quotas). If that fails, download the "img_align_celeba" folder
# manually, e.g. from Kaggle (dataset "jessicali9530/celeba-dataset"), and unpack it anywhere under DATA_ROOT.
from functools import lru_cache


@lru_cache(maxsize=None)
def celeba_files():
    files = sorted(glob.glob(os.path.join(DATA_ROOT, "**", "img_align_celeba", "*.jpg"), recursive=True))
    # if not files:
    #     try:
    #         tv.datasets.CelebA(DATA_ROOT, download=True)
    #     except Exception as e:
    #         print("torchvision download failed:", repr(e)[:200])
    #     files = sorted(glob.glob(os.path.join(DATA_ROOT, "**", "img_align_celeba", "*.jpg"), recursive=True))
    if not files:
        raise RuntimeError("CelebA images not found. Download img_align_celeba (see the comment above) into DATA_ROOT.")
    return tuple(files)


def celeba_split(kind):
    """Official CelebA partition by file number: train 1-162770, valid 162771-182637, test 182638-202599."""
    number = lambda f: int(os.path.basename(f)[:-4])
    lo, hi = {"train": (1, 162770), "valid": (162771, 182637), "test": (182638, 202599)}[kind]
    files = [f for f in celeba_files() if lo <= number(f) <= hi]
    limit = {"train": MAX_TRAIN_IMAGES, "valid": MAX_VALID_IMAGES}.get(kind)
    return files[:limit] if limit else files


class CelebAFolder(torch.utils.data.Dataset):
    def __init__(self, files, transform):
        self.files, self.transform = files, transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        return self.transform(Image.open(self.files[i]).convert("RGB")), 0   # (image, dummy label)


print(f"found {len(celeba_files())} CelebA images; train split used: {len(celeba_split('train'))}")

In [ ]:
# ResNet-18 style encoder and decoder. They are conceptually similar to a UNet,
# except that there are **no skip connections** between the two networks.
import torch.nn as nn


class EncoderBlock(nn.Module):
    def __init__(self, c_in, c_out, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(c_in, c_out, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(c_out)
        self.conv2 = nn.Conv2d(c_out, c_out, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(c_out)
        self.shortcut = nn.Identity() if stride == 1 and c_in == c_out else nn.Sequential(
            nn.Conv2d(c_in, c_out, 1, stride, bias=False), nn.BatchNorm2d(c_out))

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        return F.relu(self.bn2(self.conv2(out)) + self.shortcut(x))


class DecoderBlock(nn.Module):
    def __init__(self, c_in, c_out, scale=1):
        super().__init__()
        self.up = nn.Upsample(scale_factor=scale, mode="nearest") if scale > 1 else nn.Identity()
        self.conv1 = nn.Conv2d(c_in, c_in, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(c_in)
        self.conv2 = nn.Conv2d(c_in, c_out, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(c_out)
        self.shortcut = nn.Identity() if scale == 1 and c_in == c_out else nn.Sequential(
            nn.Conv2d(c_in, c_out, 1, bias=False), nn.BatchNorm2d(c_out))

    def forward(self, x):
        x = self.up(x)
        out = F.relu(self.bn1(self.conv1(x)))
        return F.relu(self.bn2(self.conv2(out)) + self.shortcut(x))


class ResNetEncoder(nn.Module):
    """image (B, 3, H, W) -> feature vector (B, 512)"""
    def __init__(self, first_conv=False, maxpool1=False):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 7, 2, 3, bias=False) if first_conv else nn.Conv2d(3, 64, 3, 1, 1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(3, 2, 1) if maxpool1 else nn.Identity())
        widths, layers, c = [64, 128, 256, 512], [], 64
        for i, w in enumerate(widths):
            layers += [EncoderBlock(c, w, stride=1 if i == 0 else 2), EncoderBlock(w, w)]; c = w
        self.layers = nn.Sequential(*layers)

    def forward(self, x):
        return torch.flatten(F.adaptive_avg_pool2d(self.layers(self.stem(x)), 1), 1)


class ResNetDecoder(nn.Module):
    """latent (B, latent_dim) -> image (B, 3, H, W), unnormalized output"""
    def __init__(self, latent_dim, input_height, first_conv=False, maxpool1=False):
        super().__init__()
        extra = int(first_conv) + int(maxpool1)                       # extra x2 upsamplings to undo the stem
        self.linear = nn.Linear(latent_dim, 512 * 4 * 4)
        self.resize = nn.Upsample(size=input_height // (8 * 2 ** extra))
        blocks = [DecoderBlock(512, 256, 2), DecoderBlock(256, 256), DecoderBlock(256, 128, 2), DecoderBlock(128, 128),
                  DecoderBlock(128, 64, 2), DecoderBlock(64, 64)]
        blocks += [DecoderBlock(64, 64, 2) for _ in range(extra)]
        self.blocks = nn.Sequential(*blocks)
        self.out = nn.Conv2d(64, 3, 3, 1, 1)

    def forward(self, z):
        x = self.resize(self.linear(z).view(-1, 512, 4, 4))
        return self.out(self.blocks(x))


def resnet18_encoder(first_conv=False, maxpool1=False):
    return ResNetEncoder(first_conv, maxpool1)


def resnet18_decoder(latent_dim, input_height, first_conv=False, maxpool1=False):
    return ResNetDecoder(latent_dim, input_height, first_conv, maxpool1)

In [ ]:
# This cell is mostly boilerplate code for working with the CelebA dataset.


class CelebABoilerplate(L.LightningModule):
    def __init__(self, image_size, batch_size=64, lr=0.0001, **kwargs):
        super().__init__(**kwargs)
        self.save_hyperparameters()

    def get_dataset(self, kind):
        return CelebAFolder(
            celeba_split(kind),
            transform=T.Compose(
                [
                    T.Resize(self.hparams.image_size),
                    T.CenterCrop(self.hparams.image_size),
                    T.ToImage(),
                    T.ToDtype(torch.float32, scale=True),
                ]
            ),
        )

    def get_dataloader(self, kind):
        return torch.utils.data.DataLoader(
            self.get_dataset(kind),
            num_workers=min(8, os.cpu_count()),
            shuffle=kind == "train",
            batch_size=self.hparams.batch_size,
        )

    def train_dataloader(self):
        return self.get_dataloader("train")

    def val_dataloader(self):
        return self.get_dataloader("valid")

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.lr)
        scheduler = step_lr_scheduler(optimizer)
        return [optimizer], [scheduler]

    def training_step(self, batch):
        return self.step(batch, kind="train")

    def validation_step(self, batch):
        return self.step(batch, kind="valid")

    def _log_loss(self, name, loss, kind):
        self.log(
            f"{kind}_{name}",
            loss,
            prog_bar=True,
            logger=True,
            on_epoch=True,
            on_step="train" in kind,
        )


def step_lr_scheduler(optimizer, step_size=3):
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=step_size)
    return {"scheduler": scheduler, "interval": "epoch", "frequency": 1}


def get_latest_version(root_dir):
    versions = sorted(glob.glob(f"{root_dir}/lightning_logs/version_*"))
    return versions[-1] if versions else None


def get_latest_checkpoint(root_dir):
    last_version = get_latest_version(root_dir)
    if last_version:
        checkpoints = f"{last_version}/checkpoints/*"
        checkpoints = sorted(glob.glob(checkpoints))
    else:
        checkpoints = []
    return checkpoints[-1] if checkpoints else None


def img_to_u8(img):
    img = img.transpose((1, 2, 0))
    img = (255 * img).round().clip(0, 255).astype(np.uint8)
    return img


def show_images(imgs, scale=2):
    imgs = np.concatenate(
        [
            np.concatenate(
                [img_to_u8(img) for img in row],
                axis=1,
            )
            for row in imgs
        ],
        axis=0,
    )

    imgs = Image.fromarray(imgs, "RGB")
    imgs = imgs.resize([scale * d for d in imgs.size])
    display(imgs)

---

## Part 1: Autoencoders (5 pt)

Autoencoders (AE, autoencoder) are a special type of neural network architecture consisting of two sequential subnetworks called the encoder and the decoder. The task of the encoder is to reduce the dimensionality of the input data, while the decoder aims to reconstruct the original data from the encoder's output.

Training an autoencoder involves minimizing the __reconstruction error__ — the distance between the original input data and the data reconstructed by the decoder. Quite often, the *distance* is measured using element-wise mean squared error, but in general, any other distance metric can be used.

If the decoder successfully reconstructs the input data, we can consider the intermediate low-dimensional data to be a special __representation__ of the original data. This compressed representation is called the *latent* representation, while the original data representation is referred to as the *natural* representation.

It's important to understand that dimensionality reduction is only possible due to *redundancy* in the natural representation of the target data. For example, color photographs of celebrity faces can naturally be represented as elements of $\mathbb{R}^{H{\times}W{\times}3}$, but this representation is clearly redundant. The vast majority of elements in $\mathbb{R}^{H{\times}W{\times}3}$ look like random noise or images of something entirely different (not celebrity faces).

On the other hand, our intuition tells us that the space of celebrity face photographs is not discrete, but possesses some notion of __locality__ (i.e., *small* changes in the pixels of a photo do not lead to drastic changes in the photo's properties). From this, it logically follows that the latent representation will likely inherit this property as well.

![autoencoder](attachment:ae.png)

### Implementation Details

Typically, when working with images, a “standard” convolutional neural network is used as the encoder, compressing the input image into a dense feature vector. The architecture of the decoder network is often a “mirrored” version of the encoder, where all layers go in reverse order, and instead of progressively reducing the resolution, the resolution is increased.


To increase the resolution of an image, layers can be replaced using the following principles: <br/>
Instead of [averaging/pooling (`AvgPool2d`)](https://pytorch.org/docs/stable/generated/torch.nn.AvgPool2d.html), use [interpolation/upscaling (`Upsample`)](https://pytorch.org/docs/stable/generated/torch.nn.Upsample.html). <br/>
Instead of [convolutions (`Conv2d`)](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)  with a stride (`stride > 1`), use [transposed convolutions  (`ConvTranspose2d`)](https://pytorch.org/docs/stable/generated/torch.nn.ConvTranspose2d.html) with the same stride.

In this and upcoming tasks, the encoder/decoder architectures use a modified version of ResNet-18.

In [ ]:
class CelebAAutoEncoder(CelebABoilerplate):
    def __init__(self, first_conv=False, maxpool1=False, latent_dim=512, **kwargs):
        super().__init__(**kwargs)
        self.save_hyperparameters()
        self.encoder = resnet18_encoder(
            self.hparams.first_conv,
            self.hparams.maxpool1,
        )
        self.decoder = resnet18_decoder(
            self.hparams.latent_dim,
            self.hparams.image_size,
            self.hparams.first_conv,
            self.hparams.maxpool1,
        )

    def forward(self, x):
        z = self.encoder(x)
        x_rec = self.decoder(z)
        return z, x_rec

    def step(self, batch, kind):
        x, _ = batch
        z, x_rec = self(x)

        # Reconstruction loss
        loss = F.mse_loss(x, x_rec)
        self._log_loss("loss", loss, kind)

        return loss

If a checkpoint for this model is in `checkpoints/` (for example, provided by your instructor), it is loaded. Otherwise the model is trained, and the result is saved there for later runs.

In [ ]:
# Create an AutoEncoder: load the checkpoint if it exists, otherwise train
if os.path.exists(ckpt_path("AutoEncoder")):
    ae = CelebAAutoEncoder.load_from_checkpoint(ckpt_path("AutoEncoder"))
else:
    ae = CelebAAutoEncoder(image_size=56)
    trainer = L.Trainer(
        callbacks=[
            L.pytorch.callbacks.DeviceStatsMonitor(),
            L.pytorch.callbacks.EarlyStopping("valid_loss", patience=2, verbose=True),
        ],
        default_root_dir="AutoEncoder",
        max_epochs=-1,
    )
    trainer.fit(ae, ckpt_path=get_latest_checkpoint("AutoEncoder"))
    trainer.save_checkpoint(ckpt_path("AutoEncoder"))

### Visualization of Results

Let's take a look at the output of the trained autoencoder. To do this, we'll visualize the original and the reconstructed images.

In [ ]:
@torch.no_grad
def show_autoencoder_results(ae, num_images=10):
    ae.train(False)
    valid_ds = ae.get_dataset("valid")

    inps = []
    outs = []
    for i in np.random.choice(len(valid_ds), size=num_images):
        inp, _ = valid_ds[i]
        _, out, *_ = ae(inp[np.newaxis].to(ae.device))
        out = out.squeeze(0).detach()
        inps.append(inp.numpy())
        outs.append(out.cpu().numpy())

    show_images([inps, outs])

In [ ]:
show_autoencoder_results(ae)

**❓Answer the question:** Do the reconstructed images look similar to the original ones?

### Properties of the Latent Space
Let's also test our hypothesis about the locality of the latent space.
To do this, we'll compute the latent representations for two randomly selected images and linearly interpolate between them to generate several intermediate latent vectors.
Then, we will decode and visualize these latent representations.

This allows us to check whether the decoder can produce meaningful and smooth transitions between images when operating on interpolated latent codes, which is a strong indicator of a well-structured and semantically meaningful latent space.

In [ ]:
@torch.no_grad
def show_autoencoder_interpolation(ae, num_images=10):
    ae.train(False)
    valid_ds = ae.get_dataset("valid")

    i0, i1 = np.random.choice(len(valid_ds), size=2)
    img0, _ = valid_ds[i0]
    img1, _ = valid_ds[i1]
    z0, *_ = ae(img0[np.newaxis].to(ae.device))
    z1, *_ = ae(img1[np.newaxis].to(ae.device))

    inps = [img0.numpy()] + (num_images - 2) * [np.zeros(img0.shape)] + [img1.numpy()]
    outs = []
    for lerp in np.linspace(0, 1, num_images):
        out = ae.decoder((1 - lerp) * z0 + lerp * z1).squeeze(0).detach()
        outs.append(out.cpu().numpy())

    show_images([inps, outs])

In [ ]:
show_autoencoder_interpolation(ae)

**❓Answer the question:** Did our assumption about the locality of the latent space hold true?

###Generating New Images with the Autoencoder

Returning to the topic of the seminar, let's consider if we can use the learned latent representation and its properties to generate new images.

As mentioned earlier, we know that the set of "celebrity face photographs" as a subspace of $\mathbb{R}^{H{\times}W{\times}3}$ represents only a tiny fraction of all possible images. Because of this, it's clear that we cannot just generate a random matrix of size $H{\times}W{\times}3$ and expect it to be a face photograph.

On the other hand, our latent space has a much smaller dimensionality, and thus it's reasonable to assume that the set of "celebrity face photographs" will be more "densely" populated within it. Let's try to visualize the results of decoding random latent vectors.

For simplicity, let's also assume that the latent vectors consist of normally distributed random variables. We will compute the mean and standard deviation of the elements of the latent vectors, generate new random latent vectors using these parameters, and then visualize the resulting images.

In [ ]:
@torch.no_grad
def get_mean_std(ae, num_samples=1000):
    ae.train(False)
    valid_ds = ae.get_dataset("valid")

    zs = []
    for i in np.random.choice(len(valid_ds), size=num_samples):
        inp, _ = valid_ds[i]
        z, *_ = ae(inp[np.newaxis].to(ae.device))
        out = z.squeeze(0)
        zs.append(out.cpu().numpy())

    mean = np.mean(zs, axis=0)
    std = np.std(zs, axis=0)
    return mean, std


@torch.no_grad
def show_decode_random(ae, mean, std, num_images=10):
    ae.train(False)

    mean = torch.from_numpy(mean).to(ae.device)
    std = torch.from_numpy(std).to(ae.device)
    imgs = []
    for i in range(num_images):
        imgs.append([])
        for k in range(num_images):
            z = mean + std * torch.randn(ae.hparams.latent_dim).to(ae.device)
            out = ae.decoder(z[np.newaxis]).squeeze(0)
            imgs[-1].append(out.cpu().numpy())

    show_images(imgs)

In [ ]:
mean, std = get_mean_std(ae)
show_decode_random(ae, mean, std)

**❓Answer the question:** Are the generated images good? If not, try to explain what the problem might be. Which of our assumptions didn't hold true (about the "locality" of the latent representation, about the "density" of the latent representation, or about the normality of the latent vectors' distribution)?

In [ ]:
# Free up some memory for the next section
del ae

---

## Part 2: Variational Autoencoders (10pt)

Variational Autoencoders (VAE) are a modification of standard autoencoders designed to improve the properties of the latent representation for generating plausible new data.

As we saw in the previous exercise, the method for generating new data using standard autoencoders did not work well because one of the assumptions we made did not hold. Specifically, the latent vectors in standard autoencoders __are not__ normally distributed.

Indeed, without any constraints on the latent representation, there is no reason to believe that the elements of the vector will follow the desired distribution. Now that we know what the issue was with the standard autoencoder, let's look at how this problem is addressed in variational autoencoders.

Instead of predicting the latent vector directly, a variational encoder predicts parameters for a distribution and then samples a random latent vector from that distribution. The resulting random latent vector, just like in a standard autoencoder, should be decoded into the original image. This ensures that the latent vectors will have the desired distribution (by construction).

![variational autoencoder](attachment:vae.png)

Conceptually, the idea of variational autoencoders is quite simple, but in practice, there are 3 important nuances that need to be considered in order to obtain a working solution.

### $N$-dimensionality of the latent vector space
Especially attentive students might have noticed that when talking about the distributions of latent representations, we intentionally omitted one important detail—the latent representations are $N$-dimensional vectors, not scalars.

In particular, this means that the elements of these vectors might not be independently distributed, and the "normal distribution" should take the form $\mathcal{N}\left(\mu,,\Sigma\right)$, where $\mu \in \mathbb{R}^N$ is the shift vector, and $\Sigma \in \mathbb{R}^{N{\times}N}$ is the covariance matrix. Using an arbitrary covariance matrix significantly complicates further conclusions, so an additional restriction is often introduced: $\Sigma = \mathrm{I}\sigma$, where $\sigma \in \mathbb{R}^N_+$ is the vector of standard deviations.

If the original matrix $\Sigma$ was diagonalizable, this restriction doesn't lead to a loss of generality (the non-diagonal multiplier can be "incorporated" into the first linear layer of the decoder).

### Regularization of the distribution

In the variational autoencoder, we replaced the prediction of specific latent values with modeling their distribution. However, in the current version of our algorithm, the encoder might "bypass" our requirement by simply predicting very large values for $\mu$ or very small values for $\sigma$. As a result, the predicted distributions would behave almost like ordinary "point" predictions in a standard autoencoder.

To prevent the degeneration of distributions, it is necessary to regularize the predicted parameters of the distributions by the encoder.


![effect of KL regularization](attachment:regularization.png)

To regularize the parameters of the distributions in the variational autoencoder, it is proposed to add a special term to the loss function that measures the degree of similarity between the predicted distribution and the standard one.

As you know, the [Kullback–Leibler (KL) divergence](https://en.wikipedia.org/wiki/Kullback%E2%80%93Leibler_divergence) defines an "analog" of the "distance" or measure of divergence between two distributions.

In this task, you are asked to analytically derive the formula for computing the Kullback–Leibler divergence between a normal distribution with parameters $\mu$, $\sigma$ (scalars) and the standard normal distribution (with parameters 0, 1):
$$
\large\mathrm{KL}\left(\mathcal{N}\left(\mu, \sigma^2\right),\,\mathcal{N}\left(0, 1\right)\right)
$$

In the course of the analytical derivation, you may need the following formulas:
- Definition of KL divergence:
  $$\large\mathrm{KL}\left(p,\,q\right) = \int_{-\infty}^{+\infty} p(x) \log\left(\frac{p(x)}{q(x)}\right) \mathrm{d}x$$
- Formula for the density of a normal distribution:
  $$\large\mathcal{N}\left(\mu, \sigma^2\right) = \frac{1}{\sigma\sqrt{2\pi}} e^{-\frac{1}{2}{\left(\frac{x-\mu}{\sigma}\right)}^2}$$
- Formula for the expectation of a continuous random variable (and its transformations):
  $$\large\mathbb{E}\left[f(x)\right] = \int_{-\infty}^{+\infty} f(x) p(x) \mathrm{d}x$$
  where $p(x)$  is the probability density function of the continuous random variable $x$, and $f(x)$ is any Borel measurable function.
- Two formulas for the variance of a random variable:
  $$\large\mathbb{D}x = \mathbb{E}\left[\left(x - \mathbb{E}x\right)^2\right] \qquad \small\text{and} \large\qquad \mathbb{D}x = \mathbb{E}\left[x^2\right] - \left(\mathbb{E}x\right)^2$$

<br/>

Using the analytically derived formula, implement the function `normal_distribution_kl_divergence`. <br/>
**Do not** use classes/functions from the `torch.distributions` module.

In [ ]:
def normal_distribution_kl_divergence(mu, sigma):
    # \/ Your code here \/
    kl_loss = (mu.square() + sigma.square() - 1) / 2 - torch.log(sigma)
    kl_loss = kl_loss.mean(axis=0).sum()
    # /\ Your code here /\
    return kl_loss

Check your implementation

In [ ]:
def test_normal_distribution_kl_divergence():
    mu_value = torch.randn(123, 321)
    sigma_value = torch.exp(torch.randn(123, 321))

    # Compute reference values
    mu_ref = torch.autograd.Variable(mu_value, requires_grad=True)
    sigma_ref = torch.autograd.Variable(sigma_value, requires_grad=True)
    zero = torch.zeros_like(mu_ref)
    one = torch.ones_like(sigma_ref)
    kl_loss_ref = torch.distributions.kl_divergence(
        torch.distributions.Normal(mu_ref, sigma_ref),
        torch.distributions.Normal(zero, one),
    )
    kl_loss_ref = kl_loss_ref.mean(axis=0).sum()
    kl_loss_ref.backward()
    mu_ref_grad = mu_ref.grad
    sigma_ref_grad = sigma_ref.grad

    # Check student implementation
    mu = torch.autograd.Variable(mu_value, requires_grad=True)
    sigma = torch.autograd.Variable(sigma_value, requires_grad=True)
    kl_loss = normal_distribution_kl_divergence(mu, sigma)

    # The function must produce the same values as the library implementation
    assert torch.allclose(kl_loss, kl_loss_ref)

    # The function must be differentiable
    assert mu.grad is None
    assert sigma.grad is None
    kl_loss.backward()
    assert torch.allclose(mu.grad, mu_ref_grad)
    assert torch.allclose(sigma.grad, sigma_ref_grad)


test_normal_distribution_kl_divergence()

### Reparameterization of Random Variables

The last nuance in implementing a variational autoencoder is in the process of sampling random variables. At this point, the forward pass for a variational autoencoder can be written as follows:

1. A feature vector $\varphi$ is extracted from the image $\mathbf{x}$  <br/>
    $\varphi = \mathrm{ENC}\left(\mathbf{x}\right)$
2. The parameters of the distribution are computed using the extracted features: <br/>
    $\mu = f_{\mu}\left(\varphi\right)$ <br/>
    $\sigma = f_{\sigma}\left(\varphi\right)$
3. A latent vector is sampled from the given distribution: <br/>
    $\mathbf{z} \sim \mathcal{N}\left(\mu,\,\sigma^2\right)$
4. The latent vector is decoded into the reconstruction of the image  $\mathbf{x}'$ <br/>
    $\mathbf{x}' = \mathrm{DEC}\left(\mathbf{z}\right)$

However, such a formulation of the forward pass does not allow us to train the encoder because the sampling operation from the distribution "$\sim$" in step 3 is not differentiable. To remove the non-differentiable operation "$\sim$" from the gradient path, we use the reparameterization trick.

Instead of directly sampling $\mathbf{z} \sim \mathcal{N}\left(\mu,\,\sigma^2\right)$, we can sample  $\mathbf{z}'$ from the standard normal distribution  $\mathbf{z}' \sim \mathcal{N}\left(0,\,1\right)$, and then [reparameterize](https://en.wikipedia.org/wiki/Normal_distribution#Generating_values_from_normal_distribution) the random value  $\mathbf{z}'$ to the desired random value $\mathbf{z}$ with the parameters $\mu$ and $\sigma$. After this substitution,  $\mathbf{z}$ should be expressed as a differentiable function of  $\mathbf{z}'$, $\sigma$ and $\mu$.

Implement the function `differentiable_sample_normal`, that performs the described computation. To sample random variables from the standard normal distribution, use the `torch.randn_like` function. Note that the logarithm of
`sigma` is passed to the function `differentiable_sample_normal` (for numerical stability and to ensure that `sigma` cannot take negative values).

In [ ]:
def differentiable_sample_normal(mu, log_sigma):
    # \/ Your code here \/
    sigma = torch.exp(log_sigma)
    random = torch.randn_like(mu)
    sample = mu + sigma * random
    # /\ Your code here /\
    return sample, sigma

Check your implementation

In [ ]:
def test_differentiable_sample_normal():
    current_seed = torch.seed()
    random_values = torch.randn(100)
    zeros_mu = torch.autograd.Variable(
        torch.zeros_like(random_values),
        requires_grad=True,
    )
    zeros_log_sigma = torch.autograd.Variable(
        torch.zeros_like(random_values),
        requires_grad=True,
    )

    torch.manual_seed(current_seed)
    sample, sigma = differentiable_sample_normal(zeros_mu, zeros_log_sigma)

    # The random values must be obtained from torch.randn/torch.randn_like
    assert torch.allclose(sample, random_values)
    # log_sigma=0 -> sigma=1
    assert torch.allclose(sigma, torch.ones_like(zeros_log_sigma))
    # The function must be differentiable
    assert zeros_mu.grad is None
    assert zeros_log_sigma.grad is None
    sample.sum().backward()
    assert torch.allclose(zeros_mu.grad, torch.ones_like(zeros_mu))
    assert torch.allclose(zeros_log_sigma.grad, random_values)


test_differentiable_sample_normal()

### Implementation Details
Carefully review the code of the provided convolutional variational autoencoder implementation. Pay special attention to the `forward` and `step` functions. Take a look at how the functions `normal_distribution_kl_divergence` and `differentiable_sample_normal` that you have implemented are used here.

In [ ]:
class CelebAVariationalAutoEncoder(CelebAAutoEncoder):
    def __init__(self, kl_coeff=0.00005, **kwargs):
        super().__init__(**kwargs)
        self.save_hyperparameters()
        self.f_mu = torch.nn.Linear(
            self.hparams.latent_dim,
            self.hparams.latent_dim,
        )
        self.f_log_sigma = torch.nn.Linear(
            self.hparams.latent_dim,
            self.hparams.latent_dim,
        )
        # initialize params so that self.f_mu(X) == X
        torch.nn.init.eye_(self.f_mu.weight)
        torch.nn.init.zeros_(self.f_mu.bias)
        # initialize params so that self.f_log_sigma(X) == -1
        torch.nn.init.zeros_(self.f_log_sigma.weight)
        torch.nn.init.constant_(self.f_log_sigma.bias, -1)

    def forward(self, x):
        features = self.encoder(x)

        mu = self.f_mu(features)
        log_sigma = self.f_log_sigma(features)

        if self.training:
            z, sigma = differentiable_sample_normal(mu, log_sigma)
        else:
            # During validation evaluate
            # encoder/decoder without noise
            z, sigma = mu, torch.exp(log_sigma)

        return z, self.decoder(z), mu, sigma

    def step(self, batch, kind):
        x, _ = batch
        z, x_rec, mu, sigma = self(x)

        # Reconstruction loss
        rec_loss = F.mse_loss(x, x_rec)
        self._log_loss("rec_loss", rec_loss, kind)

        # KL loss
        kl_loss = self.hparams.kl_coeff * normal_distribution_kl_divergence(mu, sigma)
        self._log_loss("kl_loss", kl_loss, kind)

        loss = rec_loss + kl_loss
        self._log_loss("loss", loss, kind)
        return loss

If a checkpoint for this model is in `checkpoints/` (for example, provided by your instructor), it is loaded. Otherwise the model is trained, and the result is saved there for later runs.

In [ ]:
# Create a VariationalAutoEncoder: load the checkpoint if it exists, otherwise fine-tune from the AutoEncoder
if os.path.exists(ckpt_path("VariationalAutoEncoder")):
    vae = CelebAVariationalAutoEncoder.load_from_checkpoint(ckpt_path("VariationalAutoEncoder"))
else:
    # Init model with the pretrained encoder/decoder from the previous part
    vae = CelebAVariationalAutoEncoder.load_from_checkpoint(ckpt_path("AutoEncoder"), strict=False)
    trainer = L.Trainer(
        callbacks=[
            L.pytorch.callbacks.DeviceStatsMonitor(),
            L.pytorch.callbacks.EarlyStopping("valid_loss", patience=1, verbose=True),
        ],
        default_root_dir="VariationalAutoEncoder",
    )
    trainer.fit(vae, ckpt_path=get_latest_checkpoint("VariationalAutoEncoder"))
    trainer.save_checkpoint(ckpt_path("VariationalAutoEncoder"))

### Visualization of Results
Let's look at the results of the trained variational autoencoder.

Compare the results with those from the previous part.

In [ ]:
show_autoencoder_results(vae)

In [ ]:
show_autoencoder_interpolation(vae)

**❓Answer the questions:** How has the reconstruction quality changed? Does the latent representation still have the property of locality?

### Generating New Images with a Variational Autoencoder
Let's try generating new images in the same way as we did in the previous part.

In [ ]:
mean, std = get_mean_std(vae)
show_decode_random(vae, mean, std)

**❓Answer the questions:** Did the variational autoencoder help eliminate artifacts when generating new images? What can you say about the quality of the generated images? Why do autoencoders and variational autoencoders generate slightly blurred images? What can be done to generate images with sharper boundaries and textures?

In [ ]:
# Free up some memory for the next section
del vae

---

## Part 2b: Making the decoder an explicit likelihood

Look again at the VAE: the decoder returns an image, and the reconstruction loss is `F.mse_loss(x, x_rec)`. Probabilistically, this is a **Gaussian decoder with a fixed variance**,
$$
p_\theta(\mathbf{x}\mid\mathbf{z})=\mathcal{N}\big(\mathbf{x}\mid\mu_\theta(\mathbf{z}),\,\sigma^2 I\big),\qquad
-\log p_\theta(\mathbf{x}\mid\mathbf{z})=\frac{1}{2\sigma^2}\sum_{i=1}^{D}(x_i-\mu_i)^2+D\log\sigma+\frac D2\log 2\pi ,
$$
where the decoder output is the **mean** $\mu_\theta(\mathbf{z})$ and σ never appears in the code. Here $D=3\cdot56\cdot56$ is the number of pixel values.

**❓Question 1.** `F.mse_loss` averages over all $D$ pixel values, and the KL term is multiplied by `kl_coeff` $=c$. Show that minimizing `mse_loss + c * KL` is equivalent to maximizing the ELBO with a Gaussian decoder of fixed variance $\sigma^2 = cD/2$. Compute σ for $c=5\cdot10^{-5}$. Is that a reasonable noise level for pixel values in $[0,1]$?

**Answer (Q1).**

### Gaussian negative log-likelihood

We now make σ a **learned parameter** and also restrict the mean to the pixel range with a sigmoid, $\mu_\theta(\mathbf z)=\mathrm{sigmoid}(\mathrm{DEC}(\mathbf z))$. Two variants:
* **global**: one scalar σ shared by all pixels;
* **diagonal**: a learned σ for every pixel and color channel, $\Sigma=\mathrm{diag}(\sigma_1^2,\dots,\sigma_D^2)$. It does not depend on $\mathbf z$.

Implement `gaussian_nll`: the negative log-likelihood of `x` under $\mathcal N(\mu,\sigma^2)$, with `log_sigma` broadcast against `mu` (a scalar or a `(3, H, W)` tensor), **summed** over all dimensions except the batch and **averaged** over the batch. Do not use `torch.distributions`.

In [ ]:
def gaussian_nll(x, mu, log_sigma):
    # \/ Your code here \/
    nll = 0.5 * np.log(2 * np.pi) + log_sigma + 0.5 * ((x - mu) / torch.exp(log_sigma)) ** 2
    nll = nll.flatten(1).sum(1).mean()
    # /\ Your code here /\
    return nll

Check your implementation

In [ ]:
def test_gaussian_nll():
    x = torch.rand(4, 3, 8, 8)
    for log_sigma_value in [torch.tensor(-1.3), torch.randn(3, 8, 8) * 0.3 - 2]:
        mu_value = torch.rand(4, 3, 8, 8)
        mu = mu_value.clone().requires_grad_(True)
        log_sigma = log_sigma_value.clone().requires_grad_(True)
        nll = gaussian_nll(x, mu, log_sigma)

        mu_ref = mu_value.clone().requires_grad_(True)
        log_sigma_ref = log_sigma_value.clone().requires_grad_(True)
        dist = torch.distributions.Normal(mu_ref, torch.exp(log_sigma_ref).expand_as(mu_ref))
        nll_ref = -dist.log_prob(x).flatten(1).sum(1).mean()

        assert torch.allclose(nll, nll_ref, atol=1e-4)
        nll.backward(); nll_ref.backward()
        assert torch.allclose(mu.grad, mu_ref.grad, atol=1e-5)
        assert torch.allclose(log_sigma.grad, log_sigma_ref.grad, atol=1e-4)
    print("gaussian_nll: OK")


test_gaussian_nll()

### A VAE with a learned decoder variance

Review the model below. Compared with `CelebAVariationalAutoEncoder`:
1. the decoder output goes through a **sigmoid** and is used as the mean;
2. the reconstruction term is the Gaussian NLL with a **learned** `log_sigma` (global or diagonal);
3. there is **no `kl_coeff`**: the loss is exactly the negative ELBO;
4. targets are **dequantized**, $x\to(255x+u)/256$ with $u\sim U[0,1)$, so that the continuous density is a valid model of 8-bit images. We can then report **bits per dimension**, $\text{bpd}=\dfrac{-\text{ELBO}+D\log256}{D\log2}$.

Both models are initialized from the VAE checkpoint of Part 2 and fine-tuned.

In [ ]:
class CelebAVAELearnedSigma(CelebAVariationalAutoEncoder):
    def __init__(self, sigma_mode="global", dequantize=True, init_sigma=0.1, **kwargs):
        super().__init__(**kwargs)
        self.save_hyperparameters()
        s = self.hparams.image_size
        shape = () if sigma_mode == "global" else (3, s, s)          # "diag": one sigma per pixel and channel
        self.log_sigma = torch.nn.Parameter(torch.full(shape, float(np.log(init_sigma))))

    def forward(self, x):
        z, x_raw, mu, sigma = super().forward(x)
        return z, torch.sigmoid(x_raw), mu, sigma                    # decoder mean in (0, 1)

    def step(self, batch, kind):
        x, _ = batch
        z, x_mean, mu, sigma = self(x)
        if self.hparams.dequantize:
            x = (x * 255 + torch.rand_like(x)) / 256
        log_sigma = self.log_sigma.clamp(min=-7.0)
        rec_nll = gaussian_nll(x, x_mean, log_sigma)
        kl = normal_distribution_kl_divergence(mu, sigma)
        loss = rec_nll + kl                                          # negative ELBO, no kl_coeff
        D = x[0].numel()
        self._log_loss("rec_nll", rec_nll, kind)
        self._log_loss("kl", kl, kind)
        self._log_loss("loss", loss, kind)
        self._log_loss("sigma_mean", torch.exp(log_sigma).mean(), kind)
        self._log_loss("bpd", (loss + D * np.log(256)) / (D * np.log(2)), kind)
        return loss

In [ ]:
VAE_SIGMA_EPOCHS = 5


def train_learned_sigma(mode):
    name = f"VAE_learned_sigma_{mode}"
    if os.path.exists(ckpt_path(name)):
        return CelebAVAELearnedSigma.load_from_checkpoint(ckpt_path(name))
    # start from the Part 2 VAE (the new log_sigma parameter is initialized fresh)
    model = CelebAVAELearnedSigma.load_from_checkpoint(ckpt_path("VariationalAutoEncoder"), strict=False, sigma_mode=mode)
    trainer = L.Trainer(
        max_epochs=VAE_SIGMA_EPOCHS,
        callbacks=[L.pytorch.callbacks.EarlyStopping("valid_loss", patience=1, verbose=True)],
        default_root_dir=name,
    )
    trainer.fit(model)
    trainer.save_checkpoint(ckpt_path(name))
    return model

In [ ]:
vae_global = train_learned_sigma("global")

In [ ]:
vae_diag = train_learned_sigma("diag")

In [ ]:
vae = CelebAVariationalAutoEncoder.load_from_checkpoint(ckpt_path("VariationalAutoEncoder"))

In [ ]:
@torch.no_grad()
def compare_reconstructions(models, num_images=8):
    valid_ds = models[0].get_dataset("valid")
    idx = np.random.choice(len(valid_ds), size=num_images, replace=False)
    x = torch.stack([valid_ds[i][0] for i in idx])
    rows = [x.numpy()]
    for m in models:
        m.train(False)
        rows.append(m(x.to(m.device))[1].clamp(0, 1).cpu().numpy())
    show_images(rows)


@torch.no_grad()
def prior_samples(m, n=8, add_noise=False, seed=0):
    m.train(False)
    torch.manual_seed(seed)
    z = torch.randn(n, m.hparams.latent_dim, device=m.device)
    out = m.decoder(z)
    if not hasattr(m, "log_sigma"):              # original VAE: raw decoder output is the mean
        return out.clamp(0, 1).cpu().numpy()
    mean = torch.sigmoid(out)
    x = mean + torch.exp(m.log_sigma.clamp(min=-7.0)) * torch.randn_like(mean) if add_noise else mean
    return x.clamp(0, 1).cpu().numpy()


print("rows: data | VAE (fixed sigma) | learned global sigma | learned diagonal sigma")
compare_reconstructions([vae, vae_global, vae_diag])

print("samples z ~ N(0, I), decoder MEANS: VAE | global | diagonal")
show_images([prior_samples(vae), prior_samples(vae_global), prior_samples(vae_diag)])

print("actual samples x ~ N(mean, sigma^2): global | diagonal")
show_images([prior_samples(vae_global, add_noise=True), prior_samples(vae_diag, add_noise=True)])

In [ ]:
@torch.no_grad()
def rms_reconstruction_error(m, num_images=500):
    m.train(False)
    loader = torch.utils.data.DataLoader(m.get_dataset("valid"), batch_size=100)
    se, n = 0.0, 0
    for x, _ in loader:
        x = x.to(m.device)
        se += ((m(x)[1] - x) ** 2).mean().item() * len(x); n += len(x)
        if n >= num_images:
            break
    return (se / n) ** 0.5


print(f"implicit sigma of the Part 2 VAE      : {np.sqrt(vae.hparams.kl_coeff * 3 * 56 * 56 / 2):.3f}")
print(f"learned global sigma                  : {vae_global.log_sigma.exp().item():.3f}")
print(f"RMS reconstruction error (global model): {rms_reconstruction_error(vae_global):.3f}")

sigma_map = vae_diag.log_sigma.clamp(min=-7.0).exp().detach().cpu()
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for c, name in enumerate(["R", "G", "B"]):
    im = axes[c].imshow(sigma_map[c], cmap="magma"); axes[c].set_title(f"learned sigma, channel {name}"); axes[c].axis("off")
    plt.colorbar(im, ax=axes[c], fraction=0.046)
axes[3].hist(sigma_map.flatten().numpy(), bins=50); axes[3].set_title("distribution of sigma_i")
plt.tight_layout(); plt.show()

**❓Answer the questions:**

2. Compare the learned global σ with the implicit σ from Question 1 and with the RMS reconstruction error. Show that, for a fixed decoder mean, the maximum-likelihood σ equals the RMS error.
3. How did the reconstructions, the prior samples and the KL term change compared with the Part 2 VAE? Explain the change in terms of the trade-off between reconstruction and KL (the "hidden β").
4. Where in the image is the learned diagonal σ large, and where is it small? Why? Is the diagonal model's bits/dim better than the global model's?
5. Why do we show decoder **means** rather than actual samples $\mathbf x\sim\mathcal N(\mu,\Sigma)$? What do the actual samples look like, and why?
6. What does the sigmoid on the mean buy us, and what does it cost? Consider pixels that are almost exactly 0 or 1 and the gradient $\partial\mu/\partial(\text{logit})$.
7. Why is dequantization needed to report bits/dim? What could go wrong with a per-pixel σ (especially one that depended on $\mathbf z$) on non-dequantized 8-bit data?

---

## Part 3: GAN (5pt)

Although historically the idea of generative adversarial networks (GANs) was developed independently of variational autoencoders, in this task, for educational purposes, we will consider GANs as a further development of variational autoencoders.

At the end of the previous part, we asked the question: "Why do autoencoders and variational autoencoders generate slightly blurred images?" One possible explanation is that when calculating the reconstruction error of the input image, we used pixel-wise distance between images. Thus, our reconstruction error does not account for spatial information, which forces the decoder to "blur" details where the spatial arrangement is uncertain.

To better understand why using pixel-wise distance between images leads to such results, let's consider a specific example. The cell below takes a face (a) and creates six distortions of it: (b) a shift by two pixels, (c) a brightness change, (d) a contrast change, (e) additive noise, (f) a blur, and (g) an occluding patch. Each distortion is blended with the original so that **all six have the same mean squared error** with respect to (a).

This contradicts our "intuitive" understanding of similarity. The shifted image (b) is nearly indistinguishable from the original to a human, while the noisy, blurred and occluded images are clearly different. (This is a classic illustration due to Wang & Bovik, *Mean squared error: love it or leave it?*, 2009.)

In [ ]:
@torch.no_grad()
def equal_mse_demo(image_size=112, target_mse=0.003, seed=0):
    torch.manual_seed(seed)
    tf = T.Compose([T.Resize(image_size), T.CenterCrop(image_size), T.ToImage(), T.ToDtype(torch.float32, scale=True)])
    x, _ = CelebAFolder(celeba_split("valid")[:1], tf)[0]

    def match(d):                                   # blend d with x so that MSE(x, result) == target_mse
        t = (target_mse / ((d - x) ** 2).mean()).sqrt()
        return x + t * (d - x)

    occluded = x.clone(); s = image_size // 4
    occluded[:, s:2 * s, s:2 * s] = 0.5
    variants = [
        torch.roll(x, shifts=(2, 2), dims=(1, 2)),                 # (b) shift
        x + 0.2,                                                    # (c) brightness
        0.5 + (x - 0.5) * 0.5,                                      # (d) contrast
        x + torch.randn_like(x),                                    # (e) noise
        F.avg_pool2d(x[None], 9, 1, 4, count_include_pad=False)[0], # (f) blur
        occluded,                                                   # (g) occlusion
    ]
    imgs = [x] + [match(v) for v in variants]
    print("MSE to the original:", [round(((v - x) ** 2).mean().item(), 4) for v in imgs[1:]])
    show_images([[v.clamp(0, 1).numpy() for v in imgs]], scale=1)


equal_mse_demo()

Now that we have addressed this drawback of the reconstruction error, it would be logical to try replacing the reconstruction error with another error function. Ideally, we would want this new error function to have "human intuition" about which features of the image are important and which are not. Of course, such a function probably doesn't exist analytically, but we can achieve a similar result by replacing "human intuition" with "statistical likelihood."

If we had a differentiable function that could assess the likelihood of any given image — in other words, the probability of finding such an image in a reference sample — we could use this function during the decoder training (the goal of the decoder would be to maximize the likelihood of the generated image). Fortunately, we already know how to obtain such a function: by training a binary classifier that distinguishes real images from "fake" ones generated by the model.

Thus, instead of $\min\ {\left\lVert\mathrm{DEC}\left(\mathbf{z}\right) - \mathbf{x}\right\rVert}^2_2$, we would look for $\max\ \mathbf{p}\left(\mathrm{DEC}\left(\mathbf{z}\right)\right)$, where $\mathbf{p}$ is the probability of belonging to the reference dataset. We will predict $\mathbf{p}$ using the encoder network with an additional linear layer and a sigmoid activation function (as in a standard binary classification task).

After this replacement, we can further simplify our architecture. Notice that, unlike the reconstruction error, maximizing the likelihood does not compare the generated image with a specific reference image $\mathbf{x}$. This means that the optimal parameters for the distribution $\mathbf{z}$ no longer depend on the input image. Considering their regularization using the Kullback–Leibler divergence, we can fix the optimal distribution $\mathbf{z} \sim \mathcal{N}\left(0,\,1\right)$, which does not depend on $\mathbf{x}$.

As a result, we have "swapped" the encoder and decoder. Now, the first network in the model is the decoder, which learns to generate plausible images from random noise, while the second network in the model is the encoder, which learns to distinguish "fake" generated images from real examples in the dataset.

In this configuration, the decoder is commonly referred to as the __generator,__ and the encoder is referred to as the __discriminator.__

![generative adversarial network](attachment:gan.png)

Generative Adversarial Networks (GANs) are a type of neural network architecture consisting of two sequential networks, called the generator and the discriminator. In GANs, the generator attempts to create data that looks plausible, while the discriminator tries to distinguish real data from "fake" data produced by the generator.

### Adversarial Nature of Training the Generator and Discriminator
An important aspect of generative adversarial networks (GANs) that has only been briefly mentioned so far is the adversarial nature of the training process. As can be seen, in the architecture described above, the generator network and the discriminator network actually have different, conflicting goals. The better the generator performs, the harder it becomes for the discriminator to distinguish fake images from real ones. This feature is precisely why this architecture is called adversarial.

Moreover, in the previous section, we mentioned that to assess the likelihood of images, we would "train a binary classifier" (discriminator), but in practice, we cannot *pre-train* and fix the discriminator. If the discriminator were fixed, due to the adversarial nature of the task, the generator would instantly overfit to this specific fixed discriminator.

To solve this problem, we need to alternate the training of the generator and the discriminator. This way, both networks are forced to continuously improve their results in a kind of "arms race." In the end, the training algorithm for the generative adversarial network can be summarized as follows:

1. Discriminator training step (generator weights are frozen)
  - A batch of real images $\mathbf{x}$ is selected

  - A batch of fake images $\mathbf{x}' = \mathrm{GEN}\left(\mathbf{z}\right)$ is generated (for random $\mathbf{z} \in \mathcal{N}\left(0, 1\right)$)

  - The discriminator predicts $\mathbf{p} = \mathrm{DIS}\left(\mathbf{x}\right)$ and $\mathbf{p}' = \mathrm{DIS}\left(\mathbf{x}'\right)$

  - Backpropagation of the binary cross-entropy error to the discriminator weights is performed. <br/> When computing cross-entropy, the reference label for $\mathbf{p}$ is $1$ (real image), and for $\mathbf{p}'$ the reference label is $0$ (fake image).

<br/>

2. Generator training step (discriminator weights are frozen)

  - A batch of fake images $\mathbf{x}' = \mathrm{GEN}\left(\mathbf{z}\right)$ is generated (for random $\mathbf{z} \in \mathcal{N}\left(0, 1\right)$)

  - The discriminator predicts $\mathbf{p}' = \mathrm{DIS}\left(\mathbf{x}'\right)$

  - Backpropagation of the binary cross-entropy error to the generator weights is performed. <br/> When computing cross-entropy, the reference label for $\mathbf{p}'$ is $1$ (real image).

Note that the reference labels for $\mathbf{p}'$ during the optimization of the discriminator and the generator are different. This is a crucial detail for understanding the dynamics of training generative adversarial models. Since the discriminator and the generator optimize different (and in some sense, even opposite) error functions, the training process of generative adversarial models __cannot be considered gradient descent.__

Many theorems, statements, and intuitions that hold true for standard neural network models (trained using gradient descent) do not hold for generative adversarial models. For example, during gradient descent training, we are used to seeing the loss function gradually decrease as training progresses, until it reaches a locally optimal value, at which point the model either "finishes" its training or starts overfitting. If the loss function increases, it means the model is diverging.

For generative adversarial models, the loss function can decrease, increase, or remain unchanged during training, and it is impossible to definitively judge the status of the training based on the dynamics of the loss function (or even individual components of the loss function). For generative adversarial models, the loss function reflects not the process of the model converging to an optimal solution, but the __balance__ between the generator and the discriminator. For example, if the discriminator and the generator improve at the same rate, the average value of the loss function will not change.

For generative adversarial models, the "optimal" solution is one in which the distribution $\mathrm{GEN}\left(\mathbf{z}\right)$ perfectly replicates the true distribution $\mathbf{x}$, and consequently, the discriminator cannot distinguish real data from generated data, i.e., $\mathrm{DIS}\left(\mathbf{x}\right) = \mathrm{DIS}\left(\mathbf{x}'\right) = \frac{1}{2}$.

### Implementation Details
Carefully review the code of the provided convolutional generative adversarial network. <br/> Pay special attention to the  `generator`, `discriminator` and `step` functions.

In [ ]:
class CelebAGenerativeAdversarialNetwork(CelebAAutoEncoder):

    # Perform the optimization ourselves
    automatic_optimization = False

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.save_hyperparameters()

        self.discriminator_fc = torch.nn.Linear(self.hparams.latent_dim, 1)

    def configure_optimizers(self):
        # Generator optimizer & scheduler
        gen_params = list(self.decoder.parameters())
        opt_gen = torch.optim.Adam(gen_params, lr=self.hparams.lr)
        sch_gen = step_lr_scheduler(opt_gen, step_size=10)

        # Discriminator optimizer & scheduler
        dis_params = []
        dis_params += list(self.encoder.parameters())
        dis_params += list(self.discriminator_fc.parameters())
        opt_dis = torch.optim.Adam(dis_params, lr=self.hparams.lr)
        sch_dis = step_lr_scheduler(opt_dis, step_size=10)

        return [opt_gen, opt_dis], [sch_gen, sch_dis]

    def generator(self, z=None, num_samples=None):
        if z is None:
            z = torch.randn(num_samples, self.hparams.latent_dim, device=self.device)
        else:
            assert num_samples is None

        # sigmoid restricts the output to range [0,1] like real images
        x_gen = torch.sigmoid(self.decoder(z))
        return z, x_gen

    def discriminator(self, x):
        features = self.encoder(x)
        logprob = self.discriminator_fc(features).squeeze(1)
        # sigmoid(logprob) == 0 means "fake image"
        # sigmoid(logprob) == 1 means "real image"
        return logprob

    def step(self, batch, kind):
        # Prepare data
        x_real, _ = batch
        num_samples = x_real.shape[0]
        gt_real = torch.ones(num_samples, device=self.device)
        gt_fake = torch.zeros(num_samples, device=self.device)
        opt_gen, opt_dis = self.optimizers()

        # ==== GENERATOR STEP ====

        # Only compute the gradients wrt the generator parameters
        with opt_gen.toggle_model():
            # Generate random fake images
            _, x_fake = self.generator(num_samples=num_samples)

            # Classify them with the discriminator
            logprob_fake = self.discriminator(x_fake)

            # Try to make the generated images appear real to the discriminator
            # Note that the gt for fake data is gt_real (1) here
            gen_loss = F.binary_cross_entropy_with_logits(logprob_fake, gt_real)

            # Optimize the generator
            if kind == "train":
                self.manual_backward(gen_loss)
                opt_gen.step()
                opt_gen.zero_grad()

            # Clear up some memory
            gen_loss = gen_loss.detach()
            x_fake = x_fake.detach()
            del logprob_fake

        # ==== DISCRIMINATOR STEP ====

        # Only compute the gradients wrt the discriminator parameters
        with opt_dis.toggle_model():
            # Classify the fake and real images with the discriminator
            logprob_fake = self.discriminator(x_fake)
            logprob_real = self.discriminator(x_real)

            # Try to make the discriminator classification accurate
            # Note that the gt for fake data is gt_fake (0) here
            dis_fake_loss = F.binary_cross_entropy_with_logits(logprob_fake, gt_fake)
            dis_real_loss = F.binary_cross_entropy_with_logits(logprob_real, gt_real)
            dis_loss = dis_real_loss + dis_fake_loss

            # Optimize the discriminator
            if kind == "train":
                self.manual_backward(dis_loss)
                opt_dis.step()
                opt_dis.zero_grad()

        # Log loss values
        self._log_loss("gen_loss", gen_loss, kind)
        self._log_loss("dis_real_loss", dis_real_loss, kind)
        self._log_loss("dis_fake_loss", dis_fake_loss, kind)
        self._log_loss("dis_loss", dis_loss, kind)
        self._log_loss("total_loss", gen_loss + dis_loss, kind)

    # Convenience method for eval
    def forward(self, *args, **kwargs):
        return self.generator(*args, **kwargs)

If a checkpoint for this model is in `checkpoints/` (for example, provided by your instructor), it is loaded. Otherwise the model is trained, and the result is saved there for later runs.

In [ ]:
# Create a GenerativeAdversarialNetwork: load the checkpoint if it exists, otherwise train
if os.path.exists(ckpt_path("GenerativeAdversarialNetwork")):
    gan = CelebAGenerativeAdversarialNetwork.load_from_checkpoint(ckpt_path("GenerativeAdversarialNetwork"))
else:
    gan = CelebAGenerativeAdversarialNetwork(image_size=56)

    # Init model with the pretrained encoder/decoder from the previous part
    vae = CelebAVariationalAutoEncoder.load_from_checkpoint(ckpt_path("VariationalAutoEncoder"))
    gan.encoder = vae.encoder
    gan.decoder = vae.decoder
    del vae

    trainer = L.Trainer(
        max_epochs=GAN_EPOCHS,
        callbacks=[
            L.pytorch.callbacks.DeviceStatsMonitor(),
            L.pytorch.callbacks.ModelCheckpoint(every_n_epochs=1, save_top_k=-1, verbose=True),
        ],
        default_root_dir="GenerativeAdversarialNetwork",
    )
    trainer.fit(gan, ckpt_path=get_latest_checkpoint("GenerativeAdversarialNetwork"))
    trainer.save_checkpoint(ckpt_path("GenerativeAdversarialNetwork"))

### Generating New Images Using a Generative Adversarial Network
Let's take a look at the results of the trained Generative Adversarial Network.

Notice that unlike autoencoders, Generative Adversarial Networks (GANs) only model the mapping from the latent space to the image space. This means that when visualizing the results of GANs, we cannot compare the generated image with a "reference."

Let’s visualize some randomly generated images and compare them with images generated using the Variational Autoencoder in the previous section.

In [ ]:
@torch.no_grad
def show_generate_random(gan, num_images=10):
    gan.train(False)

    _, imgs = gan.generator(num_samples=num_images * num_images)
    imgs = imgs.cpu().numpy()
    imgs = imgs.reshape((num_images, num_images) + imgs.shape[1:])

    show_images(imgs)

In [ ]:
show_generate_random(gan)

**❓Answer the questions:** Were you able to make the generated images less blurry using the Generative Adversarial Network? Did the "realism" of the generated faces change as a result? Can you explain why this is happening? What can be done to fix this issue?

**❓Answer the questions:** If you look closely at the generated images, you may notice that some faces appear multiple times (with slight variations). Can you explain why this is happening? What can be done to fix this issue?

### Bonus (0pt): Interpolation between generated images
Although we do not know the "reference" images for each latent vector, we can still try to linearly interpolate between two random vectors in the latent space. Look at the results of interpolation for several pairs of random latent vectors.

In [ ]:
@torch.no_grad
def show_gan_interpolation(gan, num_images=10):
    gan.train(False)

    z0 = torch.randn(num_images, gan.hparams.latent_dim, device=gan.device)
    z1 = torch.randn(num_images, gan.hparams.latent_dim, device=gan.device)

    outs = []
    for lerp in np.linspace(0, 1, num_images):
        z = (1 - lerp) * z0 + lerp * z1
        _, out = gan.generator(z=z)
        outs.append(out.cpu().numpy())

    show_images(np.array(outs).swapaxes(0, 1))

In [ ]:
show_gan_interpolation(gan)

**❓Answer the questions:** Do you think the latent representation still has the property of locality? How can you explain this effect?